In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🤖 AI Model Governance & Responsible AI Guide

**Managing model lifecycle, governance, fairness, bias detection, and responsible AI practices**

This guide covers:
- Model governance frameworks
- Model cards and documentation
- Bias and fairness metrics
- Model registry and versioning
- Responsible AI principles
- Model monitoring and degradation
- Explainability and interpretability
- Ethical AI governance
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Model Governance Framework

### Model Cards
```python
from dataclasses import dataclass
from typing import Dict, List, Optional
from datetime import datetime

@dataclass
class ModelCard:
    """Model card for AI transparency"""
    model_name: str
    model_version: str
    description: str
    created_date: datetime
    created_by: str
    intended_use: str
    limitations: List[str]
    performance_metrics: Dict
    bias_analysis: Dict
    fairness_metrics: Dict
    training_data_description: str
    model_architecture: str
    dependencies: List[str]
    ethical_considerations: str
    
    def to_dict(self) -> Dict:
        """Export model card"""
        return {
            'model_name': self.model_name,
            'version': self.model_version,
            'description': self.description,
            'created': self.created_date.isoformat(),
            'created_by': self.created_by,
            'intended_use': self.intended_use,
            'limitations': self.limitations,
            'performance': self.performance_metrics,
            'bias_analysis': self.bias_analysis,
            'fairness': self.fairness_metrics,
            'training_data': self.training_data_description,
            'ethical_considerations': self.ethical_considerations
        }

class ModelRegistry:
    """Central registry for all models"""
    
    def __init__(self):
        self.models: Dict[str, Dict] = {}
        self.model_cards: Dict[str, ModelCard] = {}
        self.version_history: Dict[str, List[str]] = {}
    
    def register_model(self, model_id: str, model_card: ModelCard, metadata: Dict):
        """Register model in registry"""
        self.models[model_id] = {
            'card': model_card,
            'metadata': metadata,
            'registered_at': datetime.now(),
            'status': 'active'
        }
        
        self.model_cards[model_id] = model_card
        
        if model_id not in self.version_history:
            self.version_history[model_id] = []
        
        self.version_history[model_id].append(model_card.model_version)
    
    def promote_model_to_production(self, model_id: str) -> bool:
        """Promote model to production"""
        if model_id not in self.models:
            return False
        
        self.models[model_id]['status'] = 'production'
        self.models[model_id]['promoted_at'] = datetime.now()
        
        return True
    
    def deprecate_model(self, model_id: str, reason: str):
        """Mark model as deprecated"""
        if model_id not in self.models:
            return
        
        self.models[model_id]['status'] = 'deprecated'
        self.models[model_id]['deprecation_reason'] = reason
        self.models[model_id]['deprecated_at'] = datetime.now()
    
    def get_model_history(self, model_id: str) -> List[str]:
        """Get version history for model"""
        return self.version_history.get(model_id, [])
    
    def search_models(self, criteria: Dict) -> List[str]:
        """Search registry by criteria"""
        results = []
        
        for model_id, model_info in self.models.items():
            matches = True
            
            for key, value in criteria.items():
                if key == 'status':
                    if model_info['status'] != value:
                        matches = False
                        break
                elif key == 'created_by':
                    if model_info['card'].created_by != value:
                        matches = False
                        break
            
            if matches:
                results.append(model_id)
        
        return results
```

### Approval & Sign-Off Workflows
```python
class ModelApprovalWorkflow:
    """Manage model approval process"""
    
    def __init__(self):
        self.approval_stages: List[Dict] = [
            {'stage': 'data_review', 'required_approvers': 1},
            {'stage': 'model_review', 'required_approvers': 2},
            {'stage': 'bias_audit', 'required_approvers': 1},
            {'stage': 'business_review', 'required_approvers': 1},
            {'stage': 'ethics_review', 'required_approvers': 1}
        ]
        self.approvals: Dict[str, Dict] = {}
    
    def submit_for_approval(self, model_id: str) -> bool:
        """Submit model for approval"""
        self.approvals[model_id] = {
            'submitted_at': datetime.now(),
            'current_stage': 0,
            'stage_approvals': {}
        }
        
        return True
    
    def add_approval(self, model_id: str, stage: str, approver: str, approved: bool, comments: str = ""):
        """Record approval"""
        if model_id not in self.approvals:
            return False
        
        if stage not in self.approvals[model_id]['stage_approvals']:
            self.approvals[model_id]['stage_approvals'][stage] = []
        
        self.approvals[model_id]['stage_approvals'][stage].append({
            'approver': approver,
            'approved': approved,
            'comments': comments,
            'timestamp': datetime.now()
        })
        
        return True
    
    def get_approval_status(self, model_id: str) -> Dict:
        """Get approval status"""
        if model_id not in self.approvals:
            return {}
        
        approval_info = self.approvals[model_id]
        
        status = {
            'submitted_at': approval_info['submitted_at'],
            'stages': []
        }
        
        for stage_def in self.approval_stages:
            stage = stage_def['stage']
            stage_approvals = approval_info['stage_approvals'].get(stage, [])
            
            approved_count = sum(1 for a in stage_approvals if a['approved'])
            required = stage_def['required_approvers']
            
            status['stages'].append({
                'stage': stage,
                'approved': approved_count >= required,
                'approvals': f"{approved_count}/{required}",
                'details': stage_approvals
            })
        
        status['ready_for_production'] = all(s['approved'] for s in status['stages'])
        
        return status
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Bias & Fairness Assessment

### Fairness Metrics
```python
class FairnessMetrics:
    """Measure model fairness"""
    
    def __init__(self):
        self.demographic_groups: Dict[str, List] = {}
        self.metrics: Dict[str, float] = {}
    
    def calculate_demographic_parity(self, predictions: List, ground_truth: List, 
                                    sensitive_attr: List) -> Dict:
        """Calculate demographic parity (equal benefit across groups)"""
        
        groups = {}
        for pred, truth, attr in zip(predictions, ground_truth, sensitive_attr):
            if attr not in groups:
                groups[attr] = {'positive': 0, 'total': 0}
            
            groups[attr]['total'] += 1
            if pred == 1:
                groups[attr]['positive'] += 1
        
        rates = {}
        for group, data in groups.items():
            rates[group] = data['positive'] / data['total'] if data['total'] > 0 else 0
        
        return rates
    
    def calculate_equalized_odds(self, predictions: List, ground_truth: List,
                                 sensitive_attr: List) -> Dict:
        """Calculate equalized odds (equal true positive and false positive rates)"""
        
        groups = {}
        for pred, truth, attr in zip(predictions, ground_truth, sensitive_attr):
            if attr not in groups:
                groups[attr] = {
                    'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0
                }
            
            if truth == 1 and pred == 1:
                groups[attr]['tp'] += 1
            elif truth == 0 and pred == 1:
                groups[attr]['fp'] += 1
            elif truth == 0 and pred == 0:
                groups[attr]['tn'] += 1
            elif truth == 1 and pred == 0:
                groups[attr]['fn'] += 1
        
        tpr = {}  # True positive rate
        fpr = {}  # False positive rate
        
        for group, counts in groups.items():
            tpr[group] = counts['tp'] / (counts['tp'] + counts['fn']) if (counts['tp'] + counts['fn']) > 0 else 0
            fpr[group] = counts['fp'] / (counts['fp'] + counts['tn']) if (counts['fp'] + counts['tn']) > 0 else 0
        
        return {'tpr': tpr, 'fpr': fpr}
    
    def calculate_fairness_score(self, predictions: List, ground_truth: List,
                                sensitive_attr: List) -> float:
        """Calculate overall fairness score (0-1)"""
        
        dp_rates = self.calculate_demographic_parity(predictions, ground_truth, sensitive_attr)
        
        # Calculate variance across groups
        rates_list = list(dp_rates.values())
        if not rates_list:
            return 1.0
        
        mean_rate = sum(rates_list) / len(rates_list)
        variance = sum((r - mean_rate) ** 2 for r in rates_list) / len(rates_list)
        std_dev = variance ** 0.5
        
        # Fairness score: lower variance = higher fairness
        # Perfect fairness (std_dev=0) = 1.0
        # High variance (std_dev=0.5) = 0.0
        fairness = max(0.0, 1.0 - (std_dev / 0.5))
        
        return fairness

class BiasDetector:
    """Detect and mitigate bias"""
    
    def __init__(self):
        self.bias_reports: List[Dict] = []
    
    def detect_systematic_bias(self, predictions: List, ground_truth: List, 
                              group_memberships: Dict) -> Dict:
        """Detect systematic bias across groups"""
        
        group_performance = {}
        
        for group_name, indices in group_memberships.items():
            correct = sum(1 for i in indices if predictions[i] == ground_truth[i])
            accuracy = correct / len(indices) if indices else 0
            
            group_performance[group_name] = accuracy
        
        # Check for significant disparities
        accuracies = list(group_performance.values())
        max_acc = max(accuracies) if accuracies else 0
        min_acc = min(accuracies) if accuracies else 0
        
        disparity = max_acc - min_acc
        
        has_bias = disparity > 0.1  # More than 10% difference
        
        return {
            'by_group': group_performance,
            'disparity': disparity,
            'has_significant_bias': has_bias,
            'worst_performing_group': min(group_performance, key=group_performance.get) if group_performance else None,
            'recommendation': 'Address bias' if has_bias else 'Acceptable'
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Model Monitoring & Responsible AI

### Model Performance Monitoring
```python
class ModelMonitor:
    """Monitor model performance in production"""
    
    def __init__(self, baseline_metrics: Dict):
        self.baseline_metrics = baseline_metrics
        self.performance_history: List[Dict] = []
        self.degradation_alerts: List[Dict] = []
    
    def record_performance(self, timestamp: datetime, metrics: Dict):
        """Record current performance metrics"""
        self.performance_history.append({
            'timestamp': timestamp,
            'metrics': metrics
        })
        
        # Check for degradation
        if self._detect_degradation(metrics):
            self.degradation_alerts.append({
                'timestamp': timestamp,
                'metrics': metrics,
                'severity': 'high'
            })
    
    def _detect_degradation(self, metrics: Dict) -> bool:
        """Check if performance degraded"""
        
        degradation_threshold = 0.05  # 5% drop
        
        for metric_name, baseline_value in self.baseline_metrics.items():
            if metric_name in metrics:
                current_value = metrics[metric_name]
                
                # For accuracy-like metrics, decrease is bad
                if metric_name in ['accuracy', 'f1_score', 'precision']:
                    if (baseline_value - current_value) / baseline_value > degradation_threshold:
                        return True
        
        return False
    
    def get_degradation_alerts(self) -> List[Dict]:
        """Get performance degradation alerts"""
        return self.degradation_alerts
    
    def recommend_retraining(self) -> Optional[str]:
        """Recommend retraining if needed"""
        if len(self.degradation_alerts) >= 3:
            return "Model performance degraded. Retraining recommended."
        
        return None

class ExplainabilityAnalyzer:
    """Analyze model explainability"""
    
    def __init__(self):
        self.feature_importance: Dict[str, float] = {}
        self.shap_values: List[Dict] = []
    
    def calculate_feature_importance(self, features: Dict, importance_scores: Dict):
        """Record feature importance"""
        self.feature_importance = importance_scores
    
    def generate_explanation(self, prediction: float, top_features: int = 5) -> Dict:
        """Generate explanation for prediction"""
        
        sorted_features = sorted(
            self.feature_importance.items(),
            key=lambda x: abs(x[1]),
            reverse=True
        )[:top_features]
        
        return {
            'prediction': prediction,
            'top_contributing_features': dict(sorted_features),
            'explanation': f"Prediction driven by: {', '.join(f[0] for f in sorted_features)}"
        }
```

### Ethical AI Principles
```python
class EthicalAIFramework:
    """Ensure ethical AI practices"""
    
    def __init__(self):
        self.principles = [
            'fairness',
            'transparency',
            'accountability',
            'robustness',
            'privacy'
        ]
        self.compliance_checks: Dict[str, bool] = {}
    
    def check_fairness(self, bias_report: Dict) -> bool:
        """Check fairness principle"""
        self.compliance_checks['fairness'] = not bias_report.get('has_significant_bias', False)
        return self.compliance_checks['fairness']
    
    def check_transparency(self, model_card: ModelCard) -> bool:
        """Check transparency principle"""
        is_transparent = (
            bool(model_card.description) and
            bool(model_card.limitations) and
            bool(model_card.ethical_considerations)
        )
        self.compliance_checks['transparency'] = is_transparent
        return is_transparent
    
    def check_accountability(self, approval_status: Dict) -> bool:
        """Check accountability principle"""
        self.compliance_checks['accountability'] = approval_status.get('ready_for_production', False)
        return self.compliance_checks['accountability']
    
    def get_ethics_compliance_score(self) -> float:
        """Calculate ethics compliance score"""
        if not self.compliance_checks:
            return 0.0
        
        passed = sum(1 for v in self.compliance_checks.values() if v)
        return passed / len(self.compliance_checks)
    
    def generate_ethics_report(self) -> Dict:
        """Generate ethics compliance report"""
        return {
            'principles': self.principles,
            'compliance': self.compliance_checks,
            'score': self.get_ethics_compliance_score(),
            'status': 'compliant' if self.get_ethics_compliance_score() >= 0.8 else 'needs_review'
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. AI Model Governance Checklist

✅ **Model Documentation**
- [ ] Model card created
- [ ] Intended use documented
- [ ] Limitations listed
- [ ] Training data described
- [ ] Ethical considerations noted

✅ **Fairness & Bias**
- [ ] Demographic parity measured
- [ ] Equalized odds assessed
- [ ] Bias in subgroups detected
- [ ] Disparities documented
- [ ] Mitigation strategies identified

✅ **Governance Process**
- [ ] Model registry maintained
- [ ] Approval workflow enforced
- [ ] Sign-offs collected
- [ ] Audit trail kept
- [ ] Version history tracked

✅ **Production Monitoring**
- [ ] Performance baseline established
- [ ] Degradation alerts active
- [ ] Retraining triggers set
- [ ] Metrics dashboard built
- [ ] Incident response ready

✅ **Ethical Standards**
- [ ] Fairness compliance checked
- [ ] Transparency verified
- [ ] Accountability assigned
- [ ] Privacy safeguards
- [ ] Ethics review conducted
</VSCode.Cell>
```